# Centaur IME: ベトナム語の並べ替えを小さい模型で

d 256・4 層の模型を 20 分学習し、候補の数ごとの点を測る（速さの目安 20 ms 以下に収めるため）。

使い方: L4 GPU を選び、「すべてのセルを実行」。最初のセルで Google ドライブにつなぐ（許可する）。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-07'
import os; os.makedirs(OUT, exist_ok=True); print('保存先', OUT)

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pyarrow==21.0.0 pypinyin==0.53.0 "opencc>=1.1,<2"

In [ ]:
!python repo/tools/fetch_source.py tatoeba-vie data/vi
!python repo/tools/fetch_source.py wiki-vi data/vi
!python repo/tools/fetch_source.py cc100-vi data/vi --lines 400000

In [ ]:
!python repo/tools/lang_measure/vi_measure.py --tatoeba data/vi/vie_sentences.tsv.bz2 --wiki $(ls data/vi/*.parquet | head -1) --out m --nbest-out nbest.jsonl
!wc -l nbest.jsonl

In [ ]:
!python repo/tools/models/train_task.py --task rr-vi --tatoeba data/vi/vie_sentences.tsv.bz2 --cc100 data/vi/cc100-vi.400000.txt --extra 400000 --batch 256 --amp --nbest-file nbest.jsonl --d 256 --layers 4 --heads 4 --minutes 20 --out "$OUT" --name rr-vi-d256
print(open(OUT + '/rr-vi-d256-result.txt', encoding='utf-8').read())

In [ ]:
from google.colab import runtime
runtime.unassign()